# Webcam Mini Project
Build your own webcam image classifier on the cifar100 dataset.

This mini project has two parts.

**Part A** is the groundwork: you build and train a simple neural network on an existing image dataset.

**Part B** puts that network to use: you capture an image with your webcam and let the network predict the photographed object. This can grow into a standalone app you publish on GitHub.

Follow the steps below in order.

## Part A
### 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras import backend as K
import tensorflow_datasets as tfds

### 2. Load and preprocess dataset
In case your RAM is too small, consider using cifar10 instead of cifar100.

In [ ]:
# try_gcs=True loads the prepared dataset from the TensorFlow Datasets
# Google Cloud Storage mirror instead of the original cs.toronto.edu
# server, which is often overloaded and slow to respond.
(ds_train, ds_test), ds_info = tfds.load(
    "cifar100",
    split=["train", "test"],
    as_supervised=True,
    batch_size=-1,
    with_info=True,
    try_gcs=True,
)

X_train, y_train = tfds.as_numpy(ds_train)
X_test, y_test = tfds.as_numpy(ds_test)

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

class_names = ds_info.features["label"].names

print("Training:", X_train.shape)
print("Test:", X_test.shape)

### 3. Build a **Simple** CNN  

*Suggestion* for the architecture:
1. Convolutional layer with 32 3 by 3 filters and ReLu activation
2. Maxpooling to halve the dimension (what should the pool_size in this case?)
3. Dropout 25%
4. Convolutional layer with 64 filters (the rest as above)
5. Maxpooling (repeat no. 2)
6. Dropout 25%
7. Convolutional layer with 128 filters (the rest as above)
8. Flatten the resulting feature map (no parameters)
9. Dense layer with 512 neurons and relu activation
10. Dropout 50%
11. Softmax dense output layer (how many neurons should it have? How many classes should be predicted?)  

Please look at the following documentation for the right syntax:
+ [**Sequential**](https://www.tensorflow.org/api_docs/python/tf/keras/models/Sequential)
+ [**Input**](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Input)
+ [**Conv2D**](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D)
+ [**MaxPool2D**](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPool2D)
+ [**Flatten**](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten)
+ [**Dense**](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)

In [ ]:
# in case you want to rebuild it, clear the previous one first:
# K.clear_session()

model = models.Sequential([
    
    layers.Input(...),
    layers.Conv2D(...),
    layers.MaxPooling2D(...),
    layers.Dropout(...),

    layers....,
    layers....,
    layers....,

    layers....,
    layers.Flatten(),
    layers.Dense(...),
    layers.Dropout(...),
    layers.Dense(...)  # output layer for 100 classes
])

In [ ]:
# model architecture summary
model...

### 4. Compile the Model  
- Use the 'adam' optimizer
- 'sparse_categorical_crossentropy' loss
- Chose 'accuracy' as the metric

The optimiser controls how the weights are updated from the gradients: `adam` is a robust default that adapts the step size per parameter. The loss is `sparse_categorical_crossentropy` because the labels are stored as single integers (0 to 99) rather than one-hot vectors; the sparse version reads those integers directly. Tracking `accuracy` gives a human-readable score alongside the loss.

In [ ]:
model.compile(optimizer=...,
              loss=...,
              metrics=[...])

### 5. Train the Model  

- select 10 epochs  
- a batch size of 64  
- use 20% of the training data for validation
The synatax can be found [**here**](https://www.tensorflow.org/api_docs/python/tf/keras/Model#fit).

In [ ]:
history = model.fit(...)

### 6. Evaluate the Model  

Don't worry about the poor accuracy at the moment.

In [ ]:
# in numbers
test_loss, test_acc = model.evaluate(..., verbose=2)
print(f'Test accuracy: {test_acc:.4f}')

In [ ]:
# in a plot
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training History')
plt.show()

In [ ]:
# Do some random predictions on some images of the test portion
idx = np.random.randint(len(X_test))
plt.imshow(np.squeeze(X_test[idx]))
plt.title(f'True label: {class_names[y_test[idx].item()]}')
plt.axis('off')
plt.show()

pred = model.predict(X_test[idx:idx+1])

In [ ]:
# Have a look at the (in)consistency of some test images
predictions = model.predict(X_test)

# Convert softmax probabilities to class indices
predicted_classes = np.argmax(predictions, axis=1)

# Compare first 10 predictions to true labels
for i in range(10):
    print(f"True: {y_test[i]}, Predicted: {predicted_classes[i]}")


## Part B
### Applying the model

In this task, you will feed your trained model a new image (ideally one you capture with your webcam) and let it predict what kind of object you photographed. This can grow into a standalone app you publish on GitHub.

### Guidelines

1. Improve the model from Part A (for example by adding another layer or two) and re-evaluate the performance.
2. Save the trained model so you do not have to retrain it every time, for example `model.save("webcam_classifier.keras")`. You will reload it with `keras.models.load_model(...)`.
3. Get an image to classify. You have two options:
   - **Webcam (recommended):** the helper tool [bonartm/imageclassifier](https://github.com/bonartm/imageclassifier) is a small standalone OpenCV script that opens your webcam and saves a snapshot. It is a separate project, not part of this repo, so run it on its own from a terminal:
     ```bash
     git clone https://github.com/bonartm/imageclassifier.git
     cd imageclassifier
     uv run --with opencv-python python src/capture.py data/captures
     ```
     A window opens showing your webcam. Press the space bar to save a shot into `data/captures/`, then press `q` to quit. `uv run --with opencv-python` provides OpenCV for that single command, so you do not need to install it yourself.
     > **Camera permissions:** if the webcam will not open, check that your terminal is allowed to use it.
     > - **macOS:** System Settings > Privacy & Security > Camera, then enable your terminal app.
     > - **Windows:** Settings > Privacy & security > Camera, turn on "Let desktop apps access your camera".
     > - **Linux:** there is usually no permission prompt. Make sure no other app is using the webcam and that your user can read the device (it should be in the `video` group for `/dev/video0`).
   - **Any image (simplest option):** the webcam step is optional. The model can classify any picture, so a phone photo or a downloaded image saved as a `.png` works just as well. Choose this option if you cannot access a webcam or prefer not to set one up.
4. Point the cells below at your saved image (replace `"my_image.png"` with your file's path), then preprocess it to match the training format: same pixel size (32 by 32), same colour format (RGB), and the same 0 to 1 normalisation.
5. Reload your saved model and let it predict what you photographed.
6. **Optional:** combine capture and prediction into a single standalone application.
7. Create a (public) repo with your app to share what you built, even if you only complete some of the points.

Before the model can classify your own picture, the image must be put into exactly the same form the network was trained on: resized to 32x32 pixels, converted to RGB, and scaled to the range 0 to 1. The cells below do this step by step.

In [ ]:
#
...
#

In [ ]:
# Load an image
from PIL import Image
img = Image.open("my_image.png").convert('RGB')

In [ ]:
# Downscale to 32x32 pixels
from tensorflow.image import resize
img_array = resize(img, [32, 32])

In [ ]:
# Convert to numpy array and normalize pixel values between 0 and 1
img_array = np.array(img_array) / 255.0 

In [ ]:
# Add batch dimension for prediction
img_array = img_array.reshape(1, 32, 32, 3)

In [ ]:
# Make a prediction
pred = model.predict(...)
predicted_class = np.argmax(...)

In [ ]:
# Visualize with class name
print("Predicted class:", predicted_class, '-', class_names[predicted_class])

plt.imshow(img)
plt.title(f"Predicted class: {class_names[predicted_class]}")
plt.axis('off')
plt.show()

In [ ]:
#
...
#

## References

- [TensorFlow CNN tutorial](https://www.tensorflow.org/tutorials/images/cnn): building and training a CNN image classifier with Keras.
- [CIFAR-100 on TensorFlow Datasets](https://www.tensorflow.org/datasets/catalog/cifar100): the dataset used here, with its classes and splits.
- [CNN Explainer](https://poloclub.github.io/cnn-explainer/): an interactive visualisation of how a CNN processes an image.
- [Keras layers documentation](https://www.tensorflow.org/api_docs/python/tf/keras/layers): reference for Conv2D, MaxPool2D, Flatten and Dense.